In [ ]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
!pip install -q mlflow dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.6 MB/s eta 0:00:00


In [1]:
import mlflow
import dagshub

dagshub.init(
    repo_owner="ansh777.tomar",
    repo_name="Youtube-Sentiment",
    mlflow=True
)
# Set or create an experiment
mlflow.set_experiment("Exp 2 - BoW vs TfIdf")

ModuleNotFoundError: No module named 'mlflow'

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [ ]:
df = pd.read_csv('/content/dataset.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [ ]:

# Step 1: Function to run the experiment
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    # Step 2: Vectorization
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_{vectorizer_name}_{ngram_range}")

# Step 6: Run experiments for BoW and TF-IDF with different n-grams
ngram_ranges = [(1, 1), (1, 2), (1, 3)]  # unigrams, bigrams, trigrams
max_features = 5000  # Example max feature size

for ngram_range in ngram_ranges:
    # BoW Experiments
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")

    # TF-IDF Experiments
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")

2026/08/17 15:40:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 1)_RandomForest at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1/runs/ed706bf354594dc5847186119d625a03
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1


2026/08/17 15:42:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TF-IDF_(1, 1)_RandomForest at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1/runs/2c56410c28a7434db713d7630067777b
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1


2026/08/17 15:43:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run TF-IDF_(1, 2)_RandomForest at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1/runs/d25e692189b6491d94da419e2e81364a
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1


2026/08/17 15:46:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run BoW_(1, 3)_RandomForest at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1/runs/ced96a6da76340e6a9682eb9b9a27d8f
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1


2026/08/17 15:48:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TF-IDF_(1, 3)_RandomForest at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1/runs/d6479f397772415ba102a222f15cafc8
🧪 View experiment at: https://dagshub.com/ansh777.tomar/Youtube-Sentiment.mlflow/#/experiments/1


In [ ]:
runs = mlflow.search_runs()

print("Number of runs:", len(runs))

runs[["run_id", "tags.mlflow.runName", "status"]]

Number of runs: 6


,run_id,tags.mlflow.runName,status
0,36238ad801c34278a582f87fee59eab9,"TF-IDF_(1, 3)_RandomForest",FINISHED
1,646709ad65c34bbe8ae0343bac1759df,"BoW_(1, 3)_RandomForest",FINISHED
2,b77dd34f2fa94c95ba8fe9b94dede058,"TF-IDF_(1, 2)_RandomForest",FINISHED
3,468f71d60cae4c61864496c4da5a926a,"BoW_(1, 2)_RandomForest",FINISHED
4,f760629049e144c197dd9192803b623c,"TF-IDF_(1, 1)_RandomForest",FINISHED
5,2e7a21b66eb74257b05d77c792a3566d,"BoW_(1, 1)_RandomForest",FINISHED


In [ ]:
experiment = mlflow.get_experiment_by_name("Exp 2 - BoW vs TfIdf")

print("Experiment ID:", experiment.experiment_id)
print("Experiment Name:", experiment.name)

Experiment ID: 1
Experiment Name: Exp 2 - BoW vs TfIdf


In [ ]:
print(mlflow.get_tracking_uri())

sqlite:////content/mlflow.db


In [ ]:
with mlflow.start_run(run_name="Test Exp 2"):
    mlflow.log_param("test", "BoW vs TF-IDF")
    mlflow.log_metric("test_accuracy", 0.90)

In [ ]:
print(mlflow.get_experiment_by_name("Exp 2 - BoW vs TfIdf"))

<Experiment: artifact_location='/content/mlruns/1', creation_time=1786979327785, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786979327785, lifecycle_stage='active', name='Exp 2 - BoW vs TfIdf', tags={}, trace_location=None, workspace='default'>


In [ ]:
print(mlflow.get_tracking_uri())

sqlite:////content/mlflow.db
